<a href="https://colab.research.google.com/github/areebaarshadqureshi/Data-Analysis-Journey-CampusX/blob/main/06-EDA-and-Data-Preprocessing%20/04-Feature_Engineering/01-Feature_Transformation/Distirbution/Column_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Why Do We Need ColumnTransformer?

In real-world datasets, different columns require different preprocessing steps.

For example:

<ul>
<li><b>Numerical columns</b> → Scaling</li>
<li><b>Categorical columns</b> → Encoding</li>
<li><b>Some columns</b> → Imputation</li>
<li><b>Some columns</b> → Dropping</li>
</ul>

Applying one preprocessing technique to the entire dataset is incorrect because each feature type requires different treatment.

That is why we use <b>ColumnTransformer</b> in Scikit-learn.

It allows us to:

<ul>
<li>Apply different transformations to different columns</li>
<li>Combine them into a single preprocessing object</li>
<li>Ensure consistent transformation during training and testing</li>
</ul>

---

### The Core Problem It Solves

### ❌ Without ColumnTransformer:

<ul>
<li>Manual preprocessing of columns separately</li>
<li>Risk of column misalignment</li>
<li>Difficult pipeline integration</li>
<li>Error-prone during deployment</li>
</ul>

### ✅ With ColumnTransformer:

<ul>
<li>Structured preprocessing</li>
<li>Fully pipeline-compatible</li>
<li>Prevents data leakage</li>
<li>Ensures reproducibility</li>
</ul>

---

### Example Scenario

Suppose we have:

<ul>
<li><b>Age</b> → Numerical → StandardScaler</li>
<li><b>Salary</b> → Numerical → StandardScaler</li>
<li><b>Gender</b> → Categorical → OneHotEncoder</li>
<li><b>City</b> → Categorical → OneHotEncoder</li>
</ul>

Mathematically, we are applying different transformation functions:

$$
X_{numeric} \rightarrow StandardScaler(X_{numeric})
$$

$$
X_{categorical} \rightarrow OneHotEncoder(X_{categorical})
$$

ColumnTransformer combines these into a single unified transformation:

$$
X \rightarrow [T_{num}(X_{num}),\; T_{cat}(X_{cat})]
$$

---

### Basic Syntax

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age', 'Salary']),
        ('cat', OneHotEncoder(drop='first'), ['Gender', 'City'])
    ]
)
````

This `preprocessor` can now be added inside a Pipeline.

### When Is It Mandatory?

<ul>
<li>When dataset has mixed data types</li>
<li>When building ML pipelines</li>
<li>When preparing models for production</li>
</ul>


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
df = pd.read_csv('/content/covid_toy.csv')

In [ ]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [ ]:
X_train

,age,gender,fever,cough,city
35,82,Female,102.0,Strong,Bangalore
89,46,Male,103.0,Strong,Bangalore
85,16,Female,103.0,Mild,Bangalore
31,83,Male,103.0,Mild,Kolkata
60,24,Female,102.0,Strong,Bangalore
...,...,...,...,...,...
43,22,Female,99.0,Mild,Bangalore
41,82,Male,NaN,Mild,Kolkata
84,69,Female,98.0,Strong,Mumbai
87,47,Male,101.0,Strong,Bangalore


1. without column transformer

In [ ]:
# adding simple imputer to fever col
si = SimpleImputer() ##all missing values replace by column mean
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [ ]:
 #Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape


(80, 1)

In [ ]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [ ]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [ ]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

2. With column transformer

In [ ]:
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [ ]:
transformer.fit_transform(X_train).shape

(80, 7)

In [ ]:
transformer.transform(X_test).shape

(20, 7)